# 딥러닝의 수치 정밀도 - FP32, FP16, BF16의 차이와 혼합 정밀도

- Tutorial ID: `cs-3-1`
- Tutorial: 딥러닝의 수치 정밀도
- Section ID: `cs-3-1-1`
- Section: FP32, FP16, BF16의 차이와 혼합 정밀도


# 062 | 딥러닝의 수치 정밀도
## FP32, FP16, BF16의 차이와 혼합 정밀도 학습

> **대상**: 딥러닝 학습 코드를 작성해본 분, `fp16=True`나 `bf16=True`가 뭔지 궁금한 분  
> **목표**: 부동소수점 표현의 원리를 이해하고, 혼합 정밀도 학습을 직접 구현한다.

---

## 📌 목차
1. 부동소수점 표현: 컴퓨터는 소수를 어떻게 저장하는가?
2. FP32, FP16, BF16 비교
3. 오버플로우와 언더플로우 문제
4. 혼합 정밀도 학습 (Mixed Precision Training)
5. Loss Scaling: FP16 학습의 핵심 트릭
6. 트랜스포머 학습에서의 정밀도 선택
7. 양자화(Quantization) 입문
8. 실습: Gradient Scaling 직접 구현

In [ ]:
import numpy as np
import struct
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

try:
    matplotlib.rc('font', family='NanumGothic')
except:
    pass
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=6, suppress=False)

# PyTorch 확인
try:
    import torch
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} 사용 가능!')
    print(f'CUDA: {torch.cuda.is_available()}')
except:
    TORCH_AVAILABLE = False
    print('PyTorch 미설치 — NumPy 시뮬레이션으로 진행합니다.')

print('\n라이브러리 로드 완료!')

---
## 1. 부동소수점 표현: 컴퓨터는 소수를 어떻게 저장하는가?

### IEEE 754 부동소수점 형식

컴퓨터는 소수를 **과학적 표기법**으로 저장합니다:

$$\text{값} = (-1)^{\text{부호}} \times 1.\text{가수부} \times 2^{\text{지수부} - \text{바이어스}}$$

```
FP32 (32비트 = 4바이트):
  [1비트 부호] [8비트 지수부] [23비트 가수부]
   S=0: 양수    바이어스=127    1.xxxxx...

예: 3.14 = 1.1001000111101011100001... × 2¹
  → 부호=0, 지수=1+127=128=10000000₂, 가수=1001000111101011100001₂
```

| 형식 | 비트 | 부호 | 지수부 | 가수부 | 범위 | 정밀도 |
|------|------|------|--------|--------|------|--------|
| **FP64** | 64 | 1 | 11 | 52 | ±1.8×10³⁰⁸ | ~15 자리 |
| **FP32** | 32 | 1 | 8 | 23 | ±3.4×10³⁸ | ~7 자리 |
| **FP16** | 16 | 1 | 5 | 10 | ±6.5×10⁴ | ~3 자리 |
| **BF16** | 16 | 1 | 8 | 7 | ±3.4×10³⁸ | ~2 자리 |

In [ ]:
# ============================================================
# 부동소수점 내부 표현 확인

In [ ]:
def float32_to_bits(x):
    """float32를 이진수로 변환"""
    # struct.pack: Python float를 4바이트로 패킹
    bytes_val = struct.pack('f', x)
    # 정수로 변환
    int_val = struct.unpack('I', bytes_val)[0]
    # 32비트 이진수 문자열
    bits = format(int_val, '032b')
    return bits

def parse_float32(x):
    """float32의 부호, 지수부, 가수부 분리"""
    bits = float32_to_bits(x)
    sign = int(bits[0])                      # 1비트 부호
    exponent_bits = bits[1:9]                # 8비트 지수부
    mantissa_bits = bits[9:]                 # 23비트 가수부
    
    exponent = int(exponent_bits, 2) - 127   # 바이어스(127) 제거
    
    # 가수부 → 실수 (암묵적 1. 포함)
    mantissa = 1.0
    for i, bit in enumerate(mantissa_bits):
        if bit == '1':
            mantissa += 2**(-(i+1))
    
    return sign, exponent, mantissa, exponent_bits, mantissa_bits

# 예제: 3.14를 FP32로 분석
test_values = [3.14, -2.5, 0.1, 1.0, 65504.0]

print('=== FP32 내부 표현 분석 ===')
for x in test_values:
    sign, exp, mant, exp_bits, mant_bits = parse_float32(x)
    bits = float32_to_bits(x)
    reconstructed = (-1)**sign * mant * 2**exp
    print(f'\n값: {x}')
    print(f'  비트: {bits[0]} | {bits[1:9]} | {bits[9:]}')
    print(f'       S   지수부({exp_bits})  가수부')
    print(f'  해석: (-1)^{sign} × {mant:.8f} × 2^{exp}')
    print(f'  복원: {reconstructed:.8f}  (오차: {abs(x-reconstructed):.2e})')

In [ ]:
# ============================================================
# FP32, FP16, BF16 특성 비교

In [ ]:
print('=== FP32, FP16, BF16 특성 비교 ===')
print()

# NumPy dtype으로 확인
dtypes = {
    'FP64 (double)': np.float64,
    'FP32 (float)': np.float32,
    'FP16 (half)': np.float16,
    # BF16은 NumPy에서 직접 미지원, 특성만 수동 계산
}

print(f'{'dtype':<18} {'바이트':>6} {'최대값':>15} {'최소양수':>12} {'엡실론':>12}')
print('-' * 68)

for name, dtype in dtypes.items():
    info = np.finfo(dtype)
    print(f'{name:<18} {info.bits//8:>5}B {info.max:>14.3e} {info.tiny:>11.3e} {info.eps:>11.3e}')

# BF16 특성 (수동)
# BF16: 부호1 + 지수부8(FP32와 동일) + 가수부7
print(f'{'BF16 (brain float)':<18} {"":>5}2B {3.4e+38:>14.3e} {1.18e-38:>11.3e} {7.8e-3:>11.3e}')

print()
print('핵심 차이:')
print('  FP16: 지수부 5비트 → 범위 좁음 (최대 65504), 정밀도 높음 (가수부 10비트)')
print('  BF16: 지수부 8비트 → 범위 FP32와 동일 (최대 3.4e38), 정밀도 낮음 (가수부 7비트)')
print('  → 딥러닝에서는 BF16이 더 안전! (overflow 위험이 낮음)')

print()
# FP16 오버플로우 데모
print('=== FP16 오버플로우 데모 ===')
values_to_test = [100.0, 1000.0, 10000.0, 65504.0, 65505.0, 100000.0]
for v in values_to_test:
    fp16_val = np.float16(v)
    fp32_val = np.float32(v)
    overflow = np.isinf(fp16_val)
    marker = ' ← 오버플로우! (inf)' if overflow else f' (오차: {abs(float(fp16_val)-v)/v*100:.2f}%)'
    print(f'  {v:>10.1f} → FP32: {fp32_val:>12.1f}, FP16: {fp16_val:>12}{marker}')

In [ ]:
# ============================================================
# 정밀도 손실 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) 각 dtype의 표현 가능 범위
ax = axes[0, 0]
formats = ['FP64', 'FP32', 'BF16', 'FP16']
max_vals = [1.8e308, 3.4e38, 3.4e38, 6.55e4]
min_pos = [2.2e-308, 1.18e-38, 1.18e-38, 6.1e-5]
bits = [64, 32, 16, 16]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']

bars = ax.barh(formats, np.log10(max_vals), color=colors, alpha=0.8)
ax.set_xlabel('log₁₀(최대값)')
ax.set_title('부동소수점 형식별 최대값 범위', fontsize=11)
for i, (bar, val, bit) in enumerate(zip(bars, max_vals, bits)):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val:.1e} ({bit}bit)', va='center', fontsize=9)
ax.set_xlim(0, max(np.log10(max_vals)) * 1.5)

# 2) 정밀도 손실 비교 (0~1 구간)
ax = axes[0, 1]
x_true = np.linspace(0, 1, 1000).astype(np.float64)
x_fp32 = x_true.astype(np.float32).astype(np.float64)
x_fp16 = x_true.astype(np.float16).astype(np.float64)

err_fp32 = np.abs(x_true - x_fp32)
err_fp16 = np.abs(x_true - x_fp16)

ax.semilogy(x_true, err_fp32 + 1e-20, 'b-', alpha=0.6, label='FP32 오차', linewidth=0.8)
ax.semilogy(x_true, err_fp16 + 1e-20, 'r-', alpha=0.6, label='FP16 오차', linewidth=0.8)
ax.set_xlabel('실제 값')
ax.set_ylabel('절대 오차 (log scale)')
ax.set_title('FP32 vs FP16 정밀도 오차\n(0~1 구간)', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# 3) 딥러닝 gradient 분포 vs FP16 표현 범위
ax = axes[1, 0]
np.random.seed(42)
# 실제 딥러닝 학습에서 나오는 gradient 분포 시뮬레이션
# 대부분이 매우 작은 값, 일부는 크다
gradients = np.concatenate([
    np.random.normal(0, 1e-4, 10000),   # 주요 gradient
    np.random.normal(0, 1e-2, 1000),    # 중간 크기
    np.random.normal(0, 1.0, 100),      # 큰 gradient
])

grad_abs = np.abs(gradients[gradients != 0])
bins = np.logspace(-8, 2, 50)
ax.hist(grad_abs, bins=bins, color='steelblue', alpha=0.7, label='Gradient 분포')
ax.axvline(x=6.1e-5, color='r', linestyle='--', linewidth=2, label='FP16 최소값 (6.1e-5)')
ax.axvline(x=6.55e4, color='orange', linestyle='--', linewidth=2, label='FP16 최대값 (65504)')
ax.set_xscale('log')
ax.set_xlabel('|Gradient| 크기 (log scale)')
ax.set_ylabel('개수')
ax.set_title('딥러닝 Gradient 크기 분포\nvs FP16 표현 가능 범위', fontsize=11)
ax.legend(fontsize=8)

underflow_ratio = (grad_abs < 6.1e-5).mean() * 100
ax.text(0.05, 0.7, f'FP16 언더플로우:\n{underflow_ratio:.1f}% gradient가\n표현 불가!',
        transform=ax.transAxes, color='red', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# 4) BF16 vs FP16 안전성 비교
ax = axes[1, 1]
categories = ['정상 범위\n(학습 가능)', 'FP16 언더플로우\n(gradient = 0)', 'FP16 오버플로우\n(inf/NaN)']
fp16_ratios = [65, 30, 5]   # 예시 비율
bf16_ratios = [95, 4, 1]

x = np.arange(len(categories))
width = 0.35

rects1 = ax.bar(x - width/2, fp16_ratios, width, label='FP16', color='#F44336', alpha=0.8)
rects2 = ax.bar(x + width/2, bf16_ratios, width, label='BF16', color='#4CAF50', alpha=0.8)

ax.set_ylabel('비율 (%)')
ax.set_title('FP16 vs BF16 학습 안정성\n(트랜스포머 학습 시)', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for rect, val in zip(list(rects1) + list(rects2), fp16_ratios + bf16_ratios):
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.5,
            f'{val}%', ha='center', va='bottom', fontsize=10)

plt.suptitle('부동소수점 형식 특성 비교', fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. 혼합 정밀도 학습 (Mixed Precision Training)

### 핵심 아이디어

**FP16으로만 학습하면:**
- 메모리 절약 ✅
- GPU Tensor Core 활용 ✅  
- Gradient 언더플로우로 학습 불안정 ❌

**혼합 정밀도 학습:**
- **순전파 (Forward)**: FP16 사용 → 빠름
- **역전파 (Backward)**: FP16 gradient 계산 → 빠름
- **가중치 업데이트 (Optimizer)**: FP32 마스터 가중치 유지 → 안전

```
┌──────────────────────────────────────────────────┐
│ 마스터 가중치 W_fp32  (FP32, 정확한 가중치 유지)   │
│          ↓ FP32→FP16 복사                        │
│ 작업 가중치 W_fp16  (FP16, 순전파/역전파에 사용)   │
│          ↓ 순전파                                 │
│ 출력 y_fp16  (FP16)                               │
│          ↓ Loss Scaling 적용                      │
│ Loss * scale_factor                               │
│          ↓ 역전파                                 │
│ Gradient_fp16 (FP16)                             │
│          ↓ Unscaling + FP32 변환                  │
│ Gradient_fp32 / scale_factor  (FP32)             │
│          ↓ 옵티마이저 업데이트                     │
│ W_fp32 ← W_fp32 - lr * Gradient_fp32            │
└──────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Loss Scaling 원리 이해

In [ ]:
print('=== Loss Scaling 원리 ===')
print()
print('문제: FP16에서 작은 gradient가 0으로 변환됨 (언더플로우)')
print('해결: Loss를 큰 값으로 스케일링 → gradient도 커짐 → 언더플로우 방지')
print()

# 예시: gradient = 1e-7 (FP16 최소값 6.1e-5보다 훨씬 작음)
true_gradient = 1e-7
scale_factor = 2**16  # 65536 (일반적으로 사용되는 초기값)

# FP16에서의 gradient
fp16_gradient = np.float16(true_gradient)
print(f'실제 gradient: {true_gradient}')
print(f'FP16 변환 후: {fp16_gradient}  (언더플로우! 0이 됨)')

# Loss Scaling 적용
scaled_gradient = true_gradient * scale_factor
fp16_scaled = np.float16(scaled_gradient)
unscaled = float(fp16_scaled) / scale_factor

print(f'\nLoss Scale = {scale_factor}')
print(f'스케일된 gradient: {scaled_gradient}')
print(f'FP16 변환: {fp16_scaled}  (이제 표현 가능!)')
print(f'언스케일 후: {unscaled}')
print(f'복원 오차: {abs(true_gradient - unscaled):.2e}')

print()
print('=== Dynamic Loss Scaling ===')
print('- 오버플로우 발생 시: scale_factor를 2로 나눔 (안전하게 축소)')
print('- N 스텝 연속 정상: scale_factor를 2로 곱함 (가능한 크게 유지)')
print(f'- 초기값: {scale_factor}, 최대: 2^24={2**24:,}')

# Dynamic Loss Scaling 시뮬레이션
class DynamicLossScaler:
    def __init__(self, init_scale=2**16, growth_factor=2.0, 
                 backoff_factor=0.5, growth_interval=2000):
        """
        동적 Loss Scaling
        
        Args:
            init_scale: 초기 스케일 팩터
            growth_factor: 정상 스텝 후 스케일 증가 배율
            backoff_factor: 오버플로우 시 스케일 감소 배율
            growth_interval: 스케일 증가를 위한 정상 스텝 수
        """
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.consecutive_normal = 0
        self.history = []
    
    def has_overflow(self, gradients):
        """gradient에 inf 또는 nan이 있는지 확인"""
        return any(np.any(np.isinf(g) | np.isnan(g)) for g in gradients)
    
    def update(self, is_overflow):
        if is_overflow:
            # 오버플로우: 스케일 줄이기
            self.scale *= self.backoff_factor
            self.consecutive_normal = 0
            print(f'    ⚠ 오버플로우! scale → {self.scale}')
        else:
            self.consecutive_normal += 1
            if self.consecutive_normal >= self.growth_interval:
                # 충분히 안정적: 스케일 늘리기
                self.scale *= self.growth_factor
                self.consecutive_normal = 0
                print(f'    ✅ 스케일 증가! scale → {self.scale}')
        
        self.history.append(self.scale)
        return self.scale

print('\n=== Dynamic Loss Scaling 시뮬레이션 ===')
scaler = DynamicLossScaler(init_scale=8.0, growth_interval=3)  # 빠른 데모용

# 시뮬레이션: 일부 스텝에서 오버플로우 발생
overflows = [False]*3 + [True] + [False]*5 + [True] + [False]*4
for step, overflow in enumerate(overflows):
    scale = scaler.update(overflow)
    status = '오버플로우' if overflow else '정상'
    print(f'  Step {step+1:2d}: {status:6s} → scale={scale}')

In [ ]:
# ============================================================
# 혼합 정밀도 학습 전체 파이프라인 시뮬레이션

In [ ]:
print('=== 혼합 정밀도 학습 전체 파이프라인 ===')
print()

class MixedPrecisionSimulator:
    """
    혼합 정밀도 학습 시뮬레이터 (NumPy 기반)
    
    실제 PyTorch에서는:
    - torch.cuda.amp.autocast() → 순전파를 FP16으로
    - torch.cuda.amp.GradScaler() → Loss Scaling 자동 관리
    """
    def __init__(self, d_in=4, d_out=4, use_fp16=True):
        self.use_fp16 = use_fp16
        self.dtype_work = np.float16 if use_fp16 else np.float32
        
        # FP32 마스터 가중치 (항상 FP32로 유지!)
        self.W_master = np.random.randn(d_in, d_out).astype(np.float32) * 0.1
        self.b_master = np.zeros(d_out, dtype=np.float32)
        
        # Loss Scaling
        self.loss_scale = 128.0
        
        # 통계
        self.losses = []
        self.grad_norms = []
        self.overflow_count = 0
    
    def forward(self, X):
        """
        순전파: 작업 정밀도(FP16)로 수행
        """
        if self.use_fp16:
            # FP32 마스터 가중치를 FP16으로 변환 후 사용
            W_work = self.W_master.astype(np.float16)
            b_work = self.b_master.astype(np.float16)
            X_work = X.astype(np.float16)
        else:
            W_work = self.W_master
            b_work = self.b_master
            X_work = X.astype(np.float32)
        
        # 선형 레이어
        self.X_work = X_work  # 역전파를 위해 저장
        return X_work @ W_work + b_work
    
    def compute_loss(self, pred, target):
        """MSE Loss"""
        diff = pred - target.astype(pred.dtype)
        loss = (diff ** 2).mean()
        return loss
    
    def backward(self, pred, target):
        """
        역전파: FP16으로 계산, FP32로 변환 후 업데이트
        """
        # gradient 계산 (FP16)
        diff = (pred - target.astype(pred.dtype)).astype(self.dtype_work)
        batch_size = pred.shape[0]
        
        # Loss Scaling 적용: gradient도 scale배 커짐
        if self.use_fp16:
            grad_output = (2 * diff / batch_size * self.loss_scale).astype(np.float16)
        else:
            grad_output = (2 * diff / batch_size).astype(np.float32)
        
        # 가중치 gradient
        dW = self.X_work.T @ grad_output
        db = grad_output.sum(axis=0)
        
        # 오버플로우 확인
        is_overflow = np.any(np.isinf(dW)) or np.any(np.isnan(dW))
        
        if is_overflow:
            self.overflow_count += 1
            self.loss_scale /= 2  # 스케일 줄이기
            return None, None, True
        
        # FP32로 변환 + Unscaling
        if self.use_fp16:
            dW_fp32 = dW.astype(np.float32) / self.loss_scale
            db_fp32 = db.astype(np.float32) / self.loss_scale
        else:
            dW_fp32, db_fp32 = dW, db
        
        return dW_fp32, db_fp32, False
    
    def update(self, dW, db, lr=0.01):
        """FP32 마스터 가중치 업데이트"""
        self.W_master -= lr * dW
        self.b_master -= lr * db


# 학습 시뮬레이션
np.random.seed(42)
d = 4
n_steps = 30

# 데이터 생성
X_data = np.random.randn(16, d).astype(np.float32)
W_true = np.random.randn(d, d).astype(np.float32)
y_data = (X_data @ W_true + np.random.randn(16, d) * 0.01).astype(np.float32)

# FP32 학습
model_fp32 = MixedPrecisionSimulator(d, d, use_fp16=False)
losses_fp32 = []
for step in range(n_steps):
    pred = model_fp32.forward(X_data)
    loss = model_fp32.compute_loss(pred, y_data)
    losses_fp32.append(float(loss))
    dW, db, overflow = model_fp32.backward(pred, y_data)
    if not overflow:
        model_fp32.update(dW, db, lr=0.005)

# FP16 혼합 정밀도 학습
model_fp16 = MixedPrecisionSimulator(d, d, use_fp16=True)
losses_fp16 = []
for step in range(n_steps):
    pred = model_fp16.forward(X_data)
    loss = model_fp16.compute_loss(pred, y_data)
    losses_fp16.append(float(loss))
    dW, db, overflow = model_fp16.backward(pred, y_data)
    if not overflow:
        model_fp16.update(dW, db, lr=0.005)

# 시각화
plt.figure(figsize=(10, 5))
plt.plot(losses_fp32, 'b-o', label='FP32 학습', markersize=4, linewidth=2)
plt.plot(losses_fp16, 'r-s', label='FP16 혼합 정밀도', markersize=4, linewidth=2, alpha=0.8)
plt.xlabel('학습 스텝')
plt.ylabel('Loss (MSE)')
plt.title('FP32 vs 혼합 정밀도(FP16) 학습 곡선 비교')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'FP32 최종 Loss: {losses_fp32[-1]:.6f}')
print(f'FP16 최종 Loss: {losses_fp16[-1]:.6f}')
print(f'FP16 오버플로우 횟수: {model_fp16.overflow_count}')

In [ ]:
# ============================================================
# PyTorch AMP (Automatic Mixed Precision) 코드 예시

In [ ]:
pytorch_amp_code = '''

In [ ]:
# PyTorch AMP (Automatic Mixed Precision) 실제 코드

In [ ]:
# pip install torch  설치 후 아래 코드를 실행하세요

import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

# 모델 정의 (예: 간단한 Transformer)
model = nn.TransformerEncoderLayer(
    d_model=512, nhead=8, dim_feedforward=2048, dropout=0.1
).cuda()  # GPU로 이동

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# GradScaler: Loss Scaling을 자동으로 관리
scaler = GradScaler(
    init_scale=2**16,       # 초기 스케일 (기본값)
    growth_factor=2.0,      # 스케일 증가 배율 (기본값)
    backoff_factor=0.5,     # 스케일 감소 배율 (기본값)
    growth_interval=2000,   # N 스텝 후 스케일 증가 (기본값)
)

# 학습 루프
for batch_idx, (inputs, targets) in enumerate(train_loader):
    inputs, targets = inputs.cuda(), targets.cuda()
    
    optimizer.zero_grad()
    
    # autocast: 이 블록 내의 연산을 자동으로 FP16으로 실행
    with autocast(dtype=torch.float16):  # BF16: dtype=torch.bfloat16
        outputs = model(inputs)
        loss = criterion(outputs, targets)  # FP32로 자동 변환
    
    # Loss Scaling 적용 후 역전파
    scaler.scale(loss).backward()
    
    # Gradient Clipping (선택사항, 권장)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # 오버플로우 확인 후 옵티마이저 업데이트
    # 오버플로우 시 업데이트 건너뜀 + 스케일 줄임
    scaler.step(optimizer)
    
    # 다음 스텝을 위한 스케일 업데이트
    scaler.update()
    
    if batch_idx % 100 == 0:
        print(f"Loss: {loss.item():.4f}, Scale: {scaler.get_scale():.0f}")

In [ ]:
# BF16 사용 (A100, RTX 3090+ GPU에서 권장)

In [ ]:
# BF16은 GradScaler 불필요! (범위가 FP32와 같아 오버플로우 없음)
for batch_idx, (inputs, targets) in enumerate(train_loader):
    inputs, targets = inputs.cuda(), targets.cuda()
    
    optimizer.zero_grad()
    
    with autocast(dtype=torch.bfloat16):  # BF16으로 전환
        outputs = model(inputs)
        loss = criterion(outputs, targets)
    
    # BF16: GradScaler 없이 그냥 backward()
    loss.backward()
    
    # Gradient Clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
'''

print('=== PyTorch AMP 코드 예시 ===')
print(pytorch_amp_code)

---
## 3. 양자화(Quantization) 입문

혼합 정밀도 학습의 다음 단계: **더 낮은 비트 수**로 모델 추론  
- **INT8 양자화**: 8비트 정수로 가중치 저장 → FP32 대비 4배 메모리 절약
- **INT4 양자화**: 4비트 정수 → 8배 메모리 절약 (LLM 압축에 핵심!)

In [ ]:
# ============================================================
# 기본 양자화 (Quantization) 구현

In [ ]:
class LinearQuantizer:
    """
    선형 양자화 (Absmax Quantization)
    
    가장 단순한 양자화 방법:
    q(x) = round(x / scale) where scale = max(|x|) / (2^(b-1) - 1)
    
    b: 비트 수 (8이면 INT8, 4면 INT4)
    """
    
    def __init__(self, bits=8):
        self.bits = bits
        self.qmax = 2**(bits-1) - 1  # INT8: 127, INT4: 7
        self.scale = None
    
    def quantize(self, x):
        """
        FP32 → INT(bits)
        """
        # 스케일 계산: 최대 절대값 기준
        self.scale = np.max(np.abs(x)) / self.qmax
        
        if self.scale == 0:
            return np.zeros_like(x, dtype=np.int8)
        
        # 양자화: 실수 → 정수
        x_q = np.round(x / self.scale).astype(np.int8)
        # 클리핑: 범위를 초과하는 값 처리
        x_q = np.clip(x_q, -self.qmax, self.qmax)
        return x_q
    
    def dequantize(self, x_q):
        """
        INT(bits) → FP32 (역양자화)
        """
        return x_q.astype(np.float32) * self.scale
    
    def quantization_error(self, x):
        """양자화 전후 오차 분석"""
        x_q = self.quantize(x)
        x_deq = self.dequantize(x_q)
        abs_err = np.abs(x - x_deq)
        return {
            'max_error': np.max(abs_err),
            'mean_error': np.mean(abs_err),
            'relative_error': np.mean(abs_err) / (np.mean(np.abs(x)) + 1e-10),
            'compression_ratio': x.itemsize / 1,  # FP32(4bytes) → INT8(1byte)
        }


# 트랜스포머 가중치 양자화 시뮬레이션
np.random.seed(42)

# 사전학습된 가중치 (정규분포 가정)
W_fp32 = np.random.randn(768, 768).astype(np.float32) * 0.02

print('=== 트랜스포머 가중치 양자화 비교 ===')
print(f'원본 가중치: FP32, shape={W_fp32.shape}, 크기={W_fp32.nbytes/1e6:.2f} MB')
print()

results = {}
for bits in [8, 4]:
    q = LinearQuantizer(bits=bits)
    err = q.quantization_error(W_fp32)
    W_q = q.quantize(W_fp32)
    W_deq = q.dequantize(W_q)
    
    # 메모리 크기 (이론적)
    mem_original = W_fp32.nbytes
    mem_quantized = W_fp32.size * (bits // 8)  # INT8: 1byte, INT4: 0.5byte
    
    results[bits] = {'err': err, 'W_deq': W_deq}
    
    print(f'INT{bits} 양자화:')
    print(f'  메모리: {mem_original/1e6:.2f} MB → {mem_quantized/1e6:.2f} MB ({mem_original/mem_quantized:.0f}배 압축)')
    print(f'  최대 오차: {err["max_error"]:.6f}')
    print(f'  평균 오차: {err["mean_error"]:.6f}')
    print(f'  상대 오차: {err["relative_error"]*100:.3f}%')
    print()

# 행렬 곱셈에서의 오차 비교
print('=== 양자화 오차가 행렬 곱셈에 미치는 영향 ===')
x_test = np.random.randn(16, 768).astype(np.float32)

output_fp32 = x_test @ W_fp32.T
for bits, result in results.items():
    output_q = x_test @ result['W_deq'].T
    err = np.mean(np.abs(output_fp32 - output_q)) / np.mean(np.abs(output_fp32)) * 100
    print(f'INT{bits}: 출력 상대 오차 = {err:.3f}%')

print()
print('▶ 실제 추론 품질 저하는 작지만, 메모리/속도 이점은 매우 큼!')
print('▶ 고급 양자화: GPTQ, AWQ, SmoothQuant 등으로 정밀도 손실 최소화')

In [ ]:
# ============================================================
# 정밀도 선택 가이드라인 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis('off')

# 표 데이터
table_data = [
    ['시나리오', 'FP32', 'FP16', 'BF16', 'INT8', 'INT4'],
    ['GPT/BERT 사전학습 (V100)', '✅ 기본', '⚠ 불안정', '❌ 미지원', '❌', '❌'],
    ['GPT/BERT 사전학습 (A100+)', '✅ 기본', '✅ 사용가능', '✅ 권장!', '❌', '❌'],
    ['파인튜닝 (LoRA)', '✅', '✅', '✅ 권장', '❌', '❌'],
    ['추론 (서비스)', '✅ 느림', '✅ 빠름', '✅ 빠름', '✅✅ 매우 빠름', '✅✅✅ 최고'],
    ['모델 저장', '✅ 기본', '✅ 절반 크기', '✅ 절반 크기', '✅ 1/4 크기', '✅ 1/8 크기'],
    ['GPU 메모리 절약', '기준', '2배', '2배', '4배', '8배'],
    ['연산 속도 (GPU)', '기준', '2-4배↑', '2-4배↑', '2-4배↑', '4-8배↑'],
    ['수치 안정성', '⭐⭐⭐⭐⭐', '⭐⭐⭐', '⭐⭐⭐⭐', '⭐⭐⭐', '⭐⭐'],
]

colors = []
for i, row in enumerate(table_data):
    row_colors = []
    for j, cell in enumerate(row):
        if i == 0:  # 헤더
            row_colors.append('#1565C0')
        elif '✅✅✅' in cell:
            row_colors.append('#1B5E20')
        elif '✅✅' in cell:
            row_colors.append('#2E7D32')
        elif '✅' in cell and '권장' in cell:
            row_colors.append('#4CAF50')
        elif '✅' in cell:
            row_colors.append('#A5D6A7')
        elif '⚠' in cell:
            row_colors.append('#FFF176')
        elif '❌' in cell:
            row_colors.append('#FFCDD2')
        elif j == 0:
            row_colors.append('#E3F2FD')
        else:
            row_colors.append('#FAFAFA')
    colors.append(row_colors)

table = ax.table(
    cellText=table_data,
    cellLoc='center',
    loc='center',
    cellColours=colors,
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.3, 2.0)

# 헤더 텍스트 색상 변경
for j in range(len(table_data[0])):
    table[(0, j)].get_text().set_color('white')
    table[(0, j)].get_text().set_fontweight('bold')

ax.set_title('딥러닝 정밀도 선택 가이드', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---
## 📚 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| **FP32** | 표준 부동소수점, 가장 안전 |
| **FP16** | 범위 좁음 → 오버플로우 위험, Loss Scaling 필요 |
| **BF16** | 범위 FP32와 동일 → 안전, A100+ GPU 권장 |
| **혼합 정밀도** | 순전파=FP16, 마스터 가중치=FP32 |
| **Loss Scaling** | FP16 언더플로우 방지: Loss × scale → gradient도 커짐 |
| **INT8/INT4** | 추론 시 메모리 절약, 학습에는 부적합 |
| **GradScaler** | PyTorch의 동적 Loss Scaling 자동화 |

### 실용적 조언
- **A100, H100 GPU**: BF16 사용 권장 (Loss Scaling 불필요, 안정적)
- **V100, T4 GPU**: FP16 + GradScaler 사용
- **모델 서빙**: INT8 또는 INT4 양자화로 메모리 절약
- **`transformers` 라이브러리**: `training_args = TrainingArguments(bf16=True)` 한 줄로 BF16 활성화

### 참고 논문
- Micikevicius et al. "Mixed Precision Training" (ICLR 2018)
- Dettmers et al. "LLM.int8()" (NeurIPS 2022)
- Frantar et al. "GPTQ" (ICLR 2023)